# ShootPX — On-Model Shots: Production Flow Notebook

This implements the **approved flow** (your flowchart) end-to-end against real fal.ai calls,
so you can run it as-is before it becomes `app/core/fal_provider.py` / `on_model_shots.py`
code. Every model is now confirmed and read from `.env` — nothing hardcoded:

- **Final generation:** Seedream 4.5 edit
- **Text-to-image** (the "generate a model" path): Seedream 4.5 text-to-image
- **VLM** (pose-prompt writing + safety checks): Gemini 2.5 Flash, run through fal's
  `openrouter/router/vision`

One place to see/swap any of them: `MODEL_CONFIG` in the Setup section below.

## The flow (mermaid — matches your diagram)

```mermaid
flowchart TD
    Start([Start: New On-Model Shot Request]) --> Source{Model Source?}

    Source -- Generate --> GenPicker[Structured picker: body type, skin tone, age bracket, build]
    GenPicker --> GenFree[Optional: free-text refinement]
    GenFree --> GenSafety1[1st safety layer: check description for nsfw/minor terms]
    GenSafety1 -- fail --> GenBlockErr[Throw error, do not generate]
    GenSafety1 -- pass --> T2I[Text-to-Image call]
    T2I --> GenNsfw2[Check generated image for nsfw]
    GenNsfw2 -- NSFW --> T2I
    GenNsfw2 -- clean --> GenPreview[Show preview to user]
    GenPreview --> GenApprove{Approve?}
    GenApprove -- No, regenerate --> T2I
    GenApprove -- Yes --> SaveLib[Save as Model Library asset]

    Source -- Upload --> Upload[Upload Model]
    Upload --> UpNsfw[Check for nsfw image]
    UpNsfw -- NSFW --> Source
    UpNsfw -- clean --> UpHost[Upload to hosted storage]

    Source -- Default --> Preset[Pick from Model Library]

    SaveLib --> ModelUrl[model_image_url]
    UpHost --> ModelUrl
    Preset --> ModelUrl

    ModelUrl --> GarmentUp[Upload Garment Photo, multi-angle allowed]
    GarmentUp --> RefQ{Add reference images?}
    RefQ -- Yes --> RefUp[Upload reference image]
    RefUp --> Assemble[Assemble ordered image list + labels]
    RefQ -- No --> Assemble

    Assemble --> CountQ{Total images <= 10?}
    CountQ -- No --> CountReject[Reject: trim reference count, suggest to user]
    CountReject --> RefQ
    CountQ -- Yes --> OutSettings[Select output settings]
    OutSettings --> AspectRes[Aspect ratio / resolution]

    AspectRes --> PromptQ{User supplied a prompt?}
    PromptQ -- Yes --> UsePrompt["Optional generation instructions\n(fidelity guardrail always prepended)"]
    PromptQ -- No --> Router[Router: garment category -> intimate or general]
    Router --> VlmPoses[VLM writes N distinct pose prompts]
    VlmPoses --> Poses["Front hero / Front 3-4 / Side 3-4 / Close product"]

    UsePrompt --> LocalSafety[Local safety pre-check: nsfw / no minors in undergarment model input]
    Poses --> LocalSafety

    LocalSafety --> PassQ{Passed?}
    PassQ -- No --> BlockReq[Block request, show error]
    PassQ -- Yes --> GenLoop[One generation call per prompt]

    GenLoop --> FalProvider[fal.ai generation provider]
    FalProvider --> SafetyChecker[enable_safety_checker: true]
    SafetyChecker --> Collect[Collect output image URLs]
    Collect --> MoreQ{More poses to generate?}
    MoreQ -- Yes --> GenLoop
    MoreQ -- No --> ReturnFinal[Return final image set to user]
    ReturnFinal --> End([End])
```

**Notes on a few edges I filled in from the diagram:** "No, regenerate" loops back to the
*Text-to-Image call* (not the structured picker) — reuses the same description. An NSFW
upload loops back to *Model Source?* — you can't regenerate an upload, only ask for a
different one. A `>10` image count rejects back to the *reference-image* step so the user
can trim, rather than aborting the whole request.

**Product review pass, 2026-08-29:** the intimate-apparel prompt-writer system prompt now
leads with garment fidelity + ecommerce framing (not just safety), and the default pose set
switched from front/back-¾/side/detail to four upper-body, product-forward angles. The
user-supplied-prompt path always prepends a fixed fidelity guardrail so a free-text
instruction can steer style but can't override garment accuracy.


## 0. Setup

Everything model-related lives in `MODEL_CONFIG`, sourced from `.env`. Change a model there — nothing else in this notebook needs to change.

In [ ]:
# pip install fal-client pillow requests python-dotenv

import os
import io
import json
import time
from pathlib import Path

import requests
from PIL import Image
from IPython.display import display
from dotenv import load_dotenv

import fal_client

load_dotenv()  # reads .env in this directory

assert os.environ.get("FAL_KEY"), "Set FAL_KEY in .env before running."

# --- MODEL_CONFIG: the ONE place to see/swap every model this flow calls. ---
# All read straight from .env, no code-side fallback drift risk. Every model is now
# confirmed — change one in .env, nothing below this cell needs to change.
MODEL_CONFIG = {
    "final_generation": os.environ["FINAL_GENERATION_MODEL"],   # Seedream 4.5 edit
    "text_to_image": os.environ["TEXT_TO_IMAGE_MODEL"],          # Seedream 4.5 text-to-image
    "vlm_endpoint": "openrouter/router/vision",                  # fal slug every VLM call below goes through
    "prompt_writer": os.environ["PROMPT_WRITER_MODEL"],          # Gemini 2.5 Flash, via vlm_endpoint
    "safety_check": os.environ["SAFETY_CHECK_MODEL"],            # Gemini 2.5 Flash, via vlm_endpoint
}
MODEL_CONFIG


## 1. Shared helpers

In [ ]:
def to_hosted_url(path_or_url: str) -> str:
    """Return a fal-hosted URL for a local file, or pass a URL through unchanged."""
    if path_or_url.startswith("http://") or path_or_url.startswith("https://"):
        return path_or_url
    return fal_client.upload_file(path_or_url)


def show(url_or_path: str, caption: str = ""):
    if url_or_path.startswith("http"):
        img = Image.open(io.BytesIO(requests.get(url_or_path, timeout=30).content))
    else:
        img = Image.open(url_or_path)
    print(caption)
    display(img)


def _strip_json_fence(text: str) -> str:
    """Gemini (and most chat models) often wrap JSON in ```json ... ``` even when told not
    to. Strip that before parsing so we don't hard-fail on the fence, not the content."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1] if "\n" in text else text[3:]
        if text.endswith("```"):
            text = text[: -3]
        text = text.strip()
        if text.lower().startswith("json"):
            text = text[4:].strip()
    return text


def _vlm_json_call(model: str, system: str, prompt: str, image_urls: list[str] | None = None, max_tokens: int = 1000) -> dict:
    """One VLM call through fal's `openrouter/router/vision` (MODEL_CONFIG['vlm_endpoint']) —
    `model` picks the underlying model (e.g. "google/gemini-2.5-flash"). Must return a bare
    JSON object/array as text; we parse `result["output"]` (fence-stripped) directly."""
    args = {
        "model": model,
        "system_prompt": system,
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": 0,
    }
    if image_urls:
        args["image_urls"] = image_urls
    result = fal_client.subscribe(MODEL_CONFIG["vlm_endpoint"], arguments=args, with_logs=True)
    return json.loads(_strip_json_fence(result["output"]))


## 2. Model Source — `generate` / `upload` / `default`

Matches the left/middle/right branches of the diagram, including both NSFW gates (on the
*description*, before spending a generation call; and on the *generated image*, before
showing a preview) and the approve/regenerate loop for the `generate` path.

In [ ]:
MODEL_LIBRARY = {
    # "preset_id": "path/or/url/to/model.jpg" — pre-vetted, skips the NSFW check below.
    "preset_1": "assets/models/preset_1.jpg",
    "preset_2": "assets/models/preset_2.jpg",
}

DESCRIPTION_BLOCKED_TERMS = ["child", "minor", "teen", "kid", "underage"]


def check_description_safety(description: str):
    """Node: '1st safety layer' — text-only, before any generation call happens."""
    text = description.lower()
    for term in DESCRIPTION_BLOCKED_TERMS:
        if term in text:
            raise ValueError(f"Blocked term '{term}' in model description — request not sent.")
    return True


def check_image_nsfw(image_url: str) -> tuple[bool, str]:
    """Node: 'Check for nsfw image' / 'Check generated image for nsfw'. VLM-based —
    model is MODEL_CONFIG['safety_check'], swap it there. Returns (is_clean, reason)."""
    result = _vlm_json_call(
        model=MODEL_CONFIG["safety_check"],
        system=(
            "You are a strict content-safety classifier for an e-commerce fashion photo "
            "pipeline. Given one image, decide if it is safe to use: it must show an adult "
            "(clearly 18+, no ambiguity), and must not be sexually explicit or otherwise "
            "inappropriate for a mainstream fashion catalog. Respond with ONLY a JSON object: "
            '{"clean": true|false, "reason": "short reason"}.'
        ),
        prompt="Classify this image per the rules in the system prompt.",
        image_urls=[image_url],
        max_tokens=200,
    )
    return bool(result["clean"]), result.get("reason", "")


def generate_model_via_text2image(description: str, image_size: str = "portrait_4_3") -> str:
    """Node: 'Text-to-Image call'."""
    prompt = (
        f"Photorealistic studio portrait of {description}. Neutral relaxed standing pose, "
        f"plain light-gray studio background, soft even lighting, natural skin texture, "
        f"looking at camera, commercial fashion-catalog quality, high detail."
    )
    result = fal_client.subscribe(
        MODEL_CONFIG["text_to_image"],
        arguments={"prompt": prompt, "image_size": image_size, "num_images": 1},
        with_logs=True,
    )
    return result["images"][0]["url"]


def resolve_model_via_generate(
    structured_attrs: dict, free_text: str = "", approve_fn=lambda url: True, max_attempts: int = 3
) -> str:
    """Nodes: structured picker -> free-text refinement -> 1st safety layer -> Text-to-Image
    call -> nsfw check -> preview -> Approve?. Loops back to Text-to-Image on either a failed
    nsfw check or a 'No, regenerate' from approve_fn, up to max_attempts.

    approve_fn(url) -> bool is the human-approval hook (the 'Show preview to user' / 'Approve?'
    step) — swap it for whatever surfaces the preview in your real UI. Defaults to auto-approve
    so this runs unattended in a notebook.
    """
    description = ", ".join(f"{k}: {v}" for k, v in structured_attrs.items())
    if free_text:
        description = f"{description}, {free_text}"
    check_description_safety(description)

    for attempt in range(1, max_attempts + 1):
        url = generate_model_via_text2image(description)
        clean, reason = check_image_nsfw(url)
        if not clean:
            print(f"[attempt {attempt}] generated image flagged nsfw ({reason}) — regenerating")
            continue
        show(url, caption=f"Preview — attempt {attempt}")
        if approve_fn(url):
            return url  # -> 'Save as Model Library asset' happens at the caller if desired
        print(f"[attempt {attempt}] not approved — regenerating")

    raise RuntimeError(f"Could not produce an approved, clean model image in {max_attempts} attempts.")


def resolve_model_via_upload(path_or_url: str) -> str:
    """Nodes: Upload Model -> Check for nsfw image -> Upload to hosted storage.
    An NSFW upload can't be regenerated — raise and let the caller ask for a different file
    ('If NSFW' loops back to Model Source? in the diagram)."""
    hosted_url = to_hosted_url(path_or_url)
    clean, reason = check_image_nsfw(hosted_url)
    if not clean:
        raise ValueError(f"Uploaded model image flagged nsfw ({reason}) — please upload a different photo.")
    return hosted_url


def resolve_model_via_default(preset_id: str) -> str:
    """Node: Pick from Model Library — presets are pre-vetted, no nsfw check needed."""
    return to_hosted_url(MODEL_LIBRARY[preset_id])


def resolve_model_image(
    mode: str,
    structured_attrs: dict | None = None,
    free_text: str = "",
    upload_path: str | None = None,
    preset_id: str | None = None,
    approve_fn=lambda url: True,
) -> str:
    """mode: 'generate' | 'upload' | 'default' — dispatches to the three branches above and
    returns model_image_url."""
    if mode == "generate":
        return resolve_model_via_generate(structured_attrs or {}, free_text, approve_fn)
    if mode == "upload":
        return resolve_model_via_upload(upload_path)
    if mode == "default":
        return resolve_model_via_default(preset_id)
    raise ValueError(f"Unknown model mode: {mode}")


## 3. Garment + reference images

Node: 'Assemble ordered image list + labels' -> 'Total images <= 10?'. Raising here (rather
than silently trimming) matches the diagram's explicit reject-and-suggest-trim step — the
caller decides how many reference images to drop and re-calls.

In [ ]:
def assemble_inputs(model_image: str, garment_images: list[str], reference_images: list[str] | None = None):
    """Returns (image_urls, labels) in order. Raises if the fal input-count ceiling (10) is
    exceeded, naming exactly how many reference images need to be trimmed."""
    reference_images = reference_images or []
    ordered = [("model", model_image)]
    ordered += [("garment", g) for g in garment_images]
    ordered += [("reference", r) for r in reference_images]

    if len(ordered) > 10:
        over_by = len(ordered) - 10
        raise ValueError(
            f"{len(ordered)} images (1 model + {len(garment_images)} garment + "
            f"{len(reference_images)} reference) exceeds the 10-image limit — "
            f"trim {over_by} reference image(s) and try again."
        )

    image_urls = [to_hosted_url(url) for _, url in ordered]
    labels = [
        f"Image {i+1} = {kind} ({'model reference' if kind == 'model' else 'exact product, preserve fidelity' if kind == 'garment' else 'style/pose reference only'})"
        for i, (kind, _) in enumerate(ordered)
    ]
    return image_urls, labels


## 4. Output settings — aspect ratio / resolution

Only `final_generation` (Seedream 4.5 edit) is wired for real generation, so this maps
straight to its `image_size` preset vocabulary — no per-provider translation needed since
there's only one provider now (unlike the model-bakeoff notebook).

In [ ]:
ASPECT_RATIO_MAP = {
    "1:1": "square_hd",
    "3:4": "portrait_4_3",
    "9:16": "portrait_16_9",
    "4:3": "landscape_4_3",
    "16:9": "landscape_16_9",
}

def build_image_size(aspect_ratio: str):
    """aspect_ratio: one of ASPECT_RATIO_MAP's keys, or a raw Seedream value
    ("auto_4K"/"auto_2K", or a {"width","height"} dict) passed straight through."""
    return ASPECT_RATIO_MAP.get(aspect_ratio, aspect_ratio)


## 5. Router + prompt logic

Node: 'Router: garment category -> intimate or general' -> picks the system prompt (and,
if you ever want it, a different `prompt_writer` model per category — the config below
supports that even though both point at the same model today).

The router itself is a plain keyword classifier on `garment_type` (the text your UI already
collects, e.g. "bra", "t-shirt") — not a separate VLM call. Cheap, deterministic, and correct
for the two categories that actually change the system prompt. Swap it for a smarter
classifier later if the keyword list stops being enough; nothing downstream needs to change
if you do.

In [ ]:
INTIMATE_GARMENT_TERMS = [
    "bra", "underwear", "lingerie", "panty", "panties", "thong", "boxer", "brief",
    "bralette", "shapewear", "corset", "negligee",
]

def classify_garment_category(garment_type: str) -> str:
    """Node: Router. Returns 'intimate' or 'general'."""
    text = garment_type.lower()
    return "intimate" if any(term in text for term in INTIMATE_GARMENT_TERMS) else "general"


PROMPT_WRITER_SYSTEM_GENERAL = """You write short, directive image-edit prompts for an \
instruction-following image editing model — not a scene-description model. Follow its native \
style: concise imperative sentences, explicit numbered image references (Image 1, Image 2, ...), \
state garment-preservation requirements once, do not pad with repeated adjectives or long \
negative-instruction lists. Each prompt must specify a distinct, named pose so a batch of \
prompts produces genuinely different shots, not just resampled variations of the same pose."""

# Garment-fidelity + ecommerce-framing focused — the garment is the commercial subject and
# takes priority over the model/background. Product review pass, 2026-08-29.
PROMPT_WRITER_SYSTEM_INTIMATE = """You are the image-direction engine for ShootPX, a professional AI ecommerce fashion photography system.

Your output is used directly as the instruction prompt for an image-generation/editing model.

GOAL:
Create commercially usable ecommerce model photography for fashion marketplaces and brand stores such as Myntra, Amazon, Flipkart, Shopify and similar retail platforms.

The supplied garment/product is the item being sold. The model exists only to present the product.

PRIORITY ORDER:
1. Product fidelity
2. Product visibility and presentation
3. Natural garment fit
4. Ecommerce composition
5. Professional photography
6. Model pose and expression
7. Background and atmosphere

PRODUCT FIDELITY:
Treat the garment reference as the authoritative source of truth.

Require the image model to preserve the supplied garment's:
- exact silhouette and proportions
- construction and geometry
- materials and texture
- colors
- patterns and prints
- seams and stitching
- trims and edges
- straps
- closures
- hardware
- logos or visible product markings

Never ask the image model to redesign, reinterpret, simplify, replace, stylize or invent product details.

The garment may be naturally fitted to the model's body, but its original design and construction must remain unchanged.

COMPOSITION:
Direct the image model toward premium ecommerce catalogue photography rather than a purely artistic fashion editorial.

The garment must be the dominant visual subject.

Prefer close or medium upper-body / half-body framing when appropriate. The product should occupy a substantial portion of the frame and remain large enough for its details to be clearly evaluated.

Keep the complete product visible whenever the requested composition allows it. Do not crop important product areas.

Keep the model's face secondary to the product.

Keep hair, hands, arms and other body elements from obscuring important product details.

PHOTOGRAPHY:
Use realistic professional commercial photography:
- controlled studio or clean lifestyle lighting
- accurate product color
- realistic skin and material texture
- natural shadows
- realistic perspective
- professional exposure
- sharp product detail
- restrained retouching
- premium retail photography quality

REFERENCE IMAGES:
Use supplied reference images as the source of truth for the person, product and any requested visual direction.

If an additional reference image is supplied, use it primarily for composition, framing, pose, lighting or photography style. Do not copy unrelated branding, text, logos, identity or other products from the reference.

POSES:
Every requested prompt must use its assigned pose distinctly.

Vary pose, camera angle and body orientation between prompts, but never sacrifice product visibility or product fidelity merely to create variety.

Do not generate poses that unnecessarily hide, cover, distort or crop the product.

INTIMATE APPAREL:
For intimate apparel, the model must be clearly an adult.

Keep the presentation professional, tasteful and suitable for mainstream retail ecommerce. Avoid sexualized, provocative or boudoir-style direction.

PROMPT STYLE:
Write concise, directive instructions for an image editing/generation model.

Do not write a long scene description.

Do not describe the model or garment when the supplied reference images already communicate those attributes.

Do not repeat the same requirement multiple times.

Use explicit image references such as Image 1, Image 2, etc., matching the input order supplied to you.

Each prompt must be self-contained and directly usable by the image-generation model.

Return only the requested JSON structure."""

PROMPT_WRITER_CONFIG = {
    "general": {
        "model": os.environ.get("PROMPT_WRITER_MODEL_GENERAL", MODEL_CONFIG["prompt_writer"]),
        "system": PROMPT_WRITER_SYSTEM_GENERAL,
    },
    "intimate": {
        "model": os.environ.get("PROMPT_WRITER_MODEL_INTIMATE", MODEL_CONFIG["prompt_writer"]),
        "system": PROMPT_WRITER_SYSTEM_INTIMATE,
    },
}

# Product review pass, 2026-08-29: back/side shots can undersell the garment for a launch
# catalog — swapped for four upper-body, product-forward angles.
DEFAULT_POSES = [
    "front-facing upper-body hero shot, garment fully visible and unobstructed",
    "front three-quarter upper-body, garment fully visible and unobstructed",
    "slight side three-quarter upper-body, garment fully visible and unobstructed",
    "close, product-focused upper-body shot emphasizing garment detail and texture",
]


def generate_pose_prompts_via_vlm(image_urls: list[str], labels: list[str], garment_type: str, num_poses: int = 4) -> list[str]:
    """Nodes: Router -> 'VLM writes N distinct pose prompts' -> the four DEFAULT_POSES."""
    category = classify_garment_category(garment_type)
    cfg = PROMPT_WRITER_CONFIG[category]
    poses = DEFAULT_POSES[:num_poses]

    prompt_text = (
        f"Images in order:\n" + "\n".join(labels) +
        f"\n\nWrite {num_poses} separate instruction-following image-edit prompts for a "
        f"premium ecommerce fashion catalog shoot of this {garment_type} on this model. One "
        f"prompt per pose, using exactly this pose list in order: {poses}. "
        f"Return ONLY a JSON array of {num_poses} strings, nothing else."
    )

    prompts = _vlm_json_call(
        model=cfg["model"], system=cfg["system"], prompt=prompt_text,
        image_urls=image_urls, max_tokens=1500,
    )
    print(f"router: garment_type={garment_type!r} -> category={category!r} (model={cfg['model']})")
    return prompts


## 6. Local safety pre-check

Node: 'Local safety pre-check: any nsfw example, no kids in undergarment model input'. Two
layers, cheapest first: a plain text blocklist on the prompt/garment description (near-zero
cost, catches the obvious), then a VLM check on the actual assembled images via
`MODEL_CONFIG['safety_check']` — this is the real gate; `enable_safety_checker=True` on the
generation call itself is a second, independent backstop, not a replacement for this step.

In [ ]:
PROMPT_BLOCKED_TERMS = ["child", "minor", "teen", "kid", "underage"]


def run_local_safety_check(prompt_text: str, image_urls: list[str]) -> tuple[bool, str]:
    """Node: 'Local safety pre-check' -> 'Passed?'. Returns (passed, reason)."""
    text = prompt_text.lower()
    for term in PROMPT_BLOCKED_TERMS:
        if term in text:
            return False, f"blocked term '{term}' in prompt text"

    result = _vlm_json_call(
        model=MODEL_CONFIG["safety_check"],
        system=(
            "You are a strict content-safety classifier for an e-commerce fashion photo "
            "pipeline about to generate an on-model shot from these input images. Check that "
            "every person shown is clearly an adult (18+, no ambiguity) and that nothing here "
            "is sexually explicit or otherwise inappropriate for a mainstream fashion catalog "
            "— this matters especially for intimate-apparel garments. Respond with ONLY a JSON "
            'object: {"pass": true|false, "reason": "short reason"}.'
        ),
        prompt="Classify these images per the rules in the system prompt.",
        image_urls=image_urls,
        max_tokens=200,
    )
    return bool(result["pass"]), result.get("reason", "")


## 7. Final generation — Seedream 4.5 edit (confirmed)

Nodes: 'One generation call per prompt' -> 'fal.ai generation provider' ->
`enable_safety_checker: true` -> 'Collect output image URLs'. Uses
`MODEL_CONFIG['final_generation']` — currently Seedream, so the argument shape below
(`image_size` preset) is Seedream's. If `FINAL_GENERATION_MODEL` in `.env` ever changes to a
model with a different schema, this function's argument-building is what needs updating —
everything upstream of it stays the same.

In [ ]:
def run_final_generation(prompt: str, image_urls: list[str], aspect_ratio: str = "3:4", num_images: int = 1, seed: int | None = None) -> list[str]:
    args = {
        "prompt": prompt,
        "image_urls": image_urls,
        "image_size": build_image_size(aspect_ratio),
        "num_images": num_images,
        "max_images": 1,
        "enable_safety_checker": True,  # do not disable — second, independent safety layer
    }
    if seed is not None:
        args["seed"] = seed

    result = fal_client.subscribe(MODEL_CONFIG["final_generation"], arguments=args, with_logs=True)
    return [img["url"] for img in result["images"]]


## 8. Orchestrator — the full flow, node for node

Wires sections 2–7 together in the exact order of the diagram: model source -> garment/refs
-> output settings -> prompt-or-router -> local safety pre-check -> generation loop ->
final image set.

In [ ]:
# UI note: label this field "Optional generation instructions", not "Prompt" — it's meant
# to steer style/scene on top of the fixed rules below, not replace them. A raw user prompt
# ("dramatic editorial pose") could otherwise silently override garment fidelity, so we always
# prepend the guardrail rather than trusting user_prompt alone.
USER_PROMPT_FIDELITY_GUARDRAIL = (
    "Preserve the exact garment shown in the reference images — shape, proportions, "
    "materials, colors, patterns, and construction details. Do not redesign or reinterpret "
    "the product. "
)


def run_on_model_shot(
    model_mode: str,                      # "generate" | "upload" | "default"
    garment_paths: list[str],
    garment_type: str = "bra",
    model_structured_attrs: dict | None = None,   # for "generate"
    model_free_text: str = "",                    # for "generate"
    model_upload_path: str | None = None,          # for "upload"
    model_preset_id: str | None = None,            # for "default"
    reference_paths: list[str] | None = None,
    aspect_ratio: str = "3:4",
    num_poses: int = 4,
    user_prompt: str | None = None,       # optional generation instructions, see guardrail above
    approve_fn=lambda url: True,          # human-approval hook, see resolve_model_via_generate
) -> dict:
    # --- Model Source? ---
    model_image = resolve_model_image(
        mode=model_mode,
        structured_attrs=model_structured_attrs,
        free_text=model_free_text,
        upload_path=model_upload_path,
        preset_id=model_preset_id,
        approve_fn=approve_fn,
    )

    # --- Garment + reference images -> Assemble ordered image list + labels -> <=10 check ---
    image_urls, labels = assemble_inputs(model_image, garment_paths, reference_paths)

    # --- User supplied a prompt? ---
    if user_prompt:
        prompts = [USER_PROMPT_FIDELITY_GUARDRAIL + user_prompt] * num_poses  # sampling variation, not true pose diversity
    else:
        prompts = generate_pose_prompts_via_vlm(image_urls, labels, garment_type, num_poses)

    # --- Local safety pre-check -> Passed? ---
    combined_prompt_text = " ".join(prompts)
    passed, reason = run_local_safety_check(combined_prompt_text, image_urls)
    if not passed:
        raise ValueError(f"Blocked at local safety pre-check: {reason}")

    # --- One generation call per prompt -> fal.ai provider -> collect -> more poses? ---
    outputs = []
    for i, p in enumerate(prompts):
        print(f"--- Pose {i+1}/{len(prompts)} ---\nPrompt: {p}\n")
        urls = run_final_generation(p, image_urls, aspect_ratio=aspect_ratio, num_images=1)
        outputs.extend(urls)
        for u in urls:
            show(u, caption=f"Pose {i+1}")

    return {"model_image": model_image, "prompts": prompts, "image_urls": image_urls, "labels": labels, "outputs": outputs}


## 9. Example run — fill in real paths/URLs before executing

In [ ]:
# result = run_on_model_shot(
#     model_mode="upload",
#     model_upload_path="assets/model_ref.jpg",
#     garment_paths=["assets/bra_front.jpg", "assets/bra_back.jpg"],
#     garment_type="bra",                 # -> router picks the "intimate" system prompt
#     reference_paths=None,
#     aspect_ratio="3:4",
#     num_poses=4,
#     user_prompt=None,                   # None => router + VLM write the 4 default poses
#                                          # (a string here is "optional generation instructions" —
#                                          # fidelity guardrail is auto-prepended, see run_on_model_shot)
# )

# "generate" path example (structured picker + free text):
# result = run_on_model_shot(
#     model_mode="generate",
#     model_structured_attrs={"body_type": "athletic", "skin_tone": "medium", "age_bracket": "25-35", "build": "average"},
#     model_free_text="warm smile, shoulder-length brown hair",
#     garment_paths=["assets/tshirt_front.jpg"],
#     garment_type="t-shirt",             # -> router picks the "general" system prompt
# )
